# The goals of this notebook:

The NOAA dataset is missing location information. Although it includes variables for state and county FIPS codes, these codes are often incorrect (at least, many/most of them don't match values I can find online - e.g. at https://transition.fcc.gov/oet/info/maps/census/fips/fips.txt)

The goal of this notebook is to add FIPS codes for each NOAA event, based on:
- The path of the event (given by begin/end latitude and longitude)
- Text in the event narrative that indicates a county or weather station
- the county names contained in the dataset

**Note:** There are roughly 50,000 entries in the NOAA data that don't appear to correspond to counties. These include maratime areas, apparent sub-regions of counties (e.g., specific mountains or beaches). I'm not really sure how to deal with these.

We will then also add variables to the NOAA data to reflect the severity of the event. This includes:
- The overall duration of the event
- The duration of the event per county???
- Predictors that are already in the NOAA data?
- Maybe some ERA5 data?


Old notes (probably delete these):
- Creating a new dataframe in which each row is a county's experience of a weather event
- Identifying the time when the weather event entered and left the county (assuming constant velocity of the weather event)
- Computing the duration of the event in each county
- Looking up the ERA5 weather data closest to the... mean time of the event?
- But we also need to keep things in context of the overall severity of the event. Maybe also compute total duration for each event, and find some way to measure total severity from ERA5 data?
- (note: this isn't the same as loading all the ERA5 data and merging that with outages... it's just augmenting NOAA instead)

First, load the NOAA data and then convert the state names to abbreviations

In [4]:
import pandas as pd

df_events = pd.read_csv("../Data/NOAA_StormEvents/StormEvents_2014_2024.csv")

In [6]:
# We'll use a dictionary to do the conversion.
# (There is a Pandas 'us' package that could do this for us, but I'm having dependency issues, so a dictionary it is...)

us_state_to_abbrev = {
    "ALABAMA": "AL",
    "ALASKA": "AK",
    "ARIZONA": "AZ",
    "ARKANSAS": "AR",
    "CALIFORNIA": "CA",
    "COLORADO": "CO",
    "CONNECTICUT": "CT",
    "DELAWARE": "DE",
    "FLORIDA": "FL",
    "GEORGIA": "GA",
    "HAWAII": "HI",
    "IDAHO": "ID",
    "ILLINOIS": "IL",
    "INDIANA": "IN",
    "IOWA": "IA",
    "KANSAS": "KS",
    "KENTUCKY": "KY",
    "LOUISIANA": "LA",
    "MAINE": "ME",
    "MARYLAND": "MD",
    "MASSACHUSETTS": "MA",
    "MICHIGAN": "MI",
    "MINNESOTA": "MN",
    "MISSISSIPPI": "MS",
    "MISSOURI": "MO",
    "MONTANA": "MT",
    "NEBRASKA": "NE",
    "NEVADA": "NV",
    "NEW HAMPSHIRE": "NH",
    "NEW JERSEY": "NJ",
    "NEW MEXICO": "NM",
    "NEW YORK": "NY",
    "NORTH CAROLINA": "NC",
    "NORTH DAKOTA": "ND",
    "OHIO": "OH",
    "OKLAHOMA": "OK",
    "OREGON": "OR",
    "PENNSYLVANIA": "PA",
    "RHODE ISLAND": "RI",
    "SOUTH CAROLINA": "SC",
    "SOUTH DAKOTA": "SD",
    "TENNESSEE": "TN",
    "TEXAS": "TX",
    "UTAH": "UT",
    "VERMONT": "VT",
    "VIRGINIA": "VA",
    "WASHINGTON": "WA",
    "WEST VIRGINIA": "WV",
    "WISCONSIN": "WI",
    "WYOMING": "WY",
    "DISTRICT OF COLUMBIA": "DC",
    "AMERICAN SAMOA": "AS",
    "GUAM": "GU",
    "NORTHERN MARIANA ISLANDS": "MP",
    "PUERTO RICO": "PR",
    "UNITED STATES MINOR OUTLYING ISLANDS": "UM",
    "U.S": "US"
}

def convert_state_name_to_abbrev(state_name):
    #Note that the state name needs to be in all capital letters
    return us_state_to_abbrev.get(state_name, "Unknown")

# Apply the function to convert the STATE variable in df_events to abbreviations
df_events['STATE_ABBREV'] = df_events['STATE'].apply(convert_state_name_to_abbrev)

Next, load the US Counties shapefile from the US Census.

This is available from https://www.census.gov/geographies/mapping-files/time-series/geo/cartographic-boundary.html

In [7]:
import geopandas

#Load the US Census Counties shapefile
counties = geopandas.read_file('../Data/cb_2023_us_county_500k')

#Concatenate STATEFP and COUNTFP and then convert to an integer
counties['FIPS'] = (counties['STATEFP'].astype(str) + counties['COUNTYFP'].astype(str)).astype(int)

Define a function that identifies all the FIPS in the path of the weather event.

This takes about 2 minutes to run.

We're going to assume that the path of the event is essentially linear. This is probably inaccurate, but we don't have any additional data (without doing something really clever with the ERA5 data) as an alternative

In [8]:
# Given a beginning point (given by BEGIN_LAT and BEGIN_LON) and an ending point (given by END_LAT and END_LON) from df_events, 
# identify which FIPS values from counties lie in the path between the beginning and ending points

def get_fips_from_path(row):
    #Get the beginning and ending values of longitude and latitude
    begin = (row['BEGIN_LON'], row['BEGIN_LAT'])
    end = (row['END_LON'], row['END_LAT'])

    #Convert them into geopandas points
    points = geopandas.points_from_xy([begin[0], end[0]], [begin[1], end[1]])

    #Get the path between the two points
    path = geopandas.GeoSeries(points)
    
    #Get the FIPS values from the counties shapefile that intersect with the path
    fips = counties[counties.geometry.intersects(path.union_all())]['FIPS'].tolist()
    
    return fips

#Apply get_fips_from_path to each row of df_events and create a new variable that lists the fips values
df_events['FIPS_from_Path'] = df_events.apply(get_fips_from_path, axis=1)

There are many rows in df_events that don't have beginning/end longitude/latitude values. However, the event_narrative variable sometimes mentions the name of a county.

The code below searches for county names in the event_narrative variable and matches them to a FIPS from the counties dataframe

Note that this takes almost 5 minutes to run

In [9]:
def get_fips_from_narrative(row):
    #Get the state from the row
    state = row['STATE_ABBREV']
    
    #Get the event narrative from the row
    narrative = row['EVENT_NARRATIVE']

    #Create a new dataframe from county_fips that has the same state
    counties_state = counties[counties['STUSPS'] == state]
    
    #Identify any word in the EVENT_NARRATIVE variable that matches the COUNTYNAME variable in county_fips for the same STATE
    fips = []
    for index, row in counties_state.iterrows():
        #if narrative is not NaN:
        if pd.notna(row['NAME']) and pd.notna(narrative):
            if row['NAME'] in narrative:
                fips.append(row['FIPS'])    
    return fips

#Apply the function to each row of df_events that lack latitude data to add a FIPS code
df_events['FIPS_from_Countyname_Narrative'] = df_events[df_events['BEGIN_LAT'].isna()].apply(get_fips_from_narrative, axis=1)

In addition to county names, sometimes the event_narrative variable refers to (what seem to be) various sorts of weather stations.

Like the county names, we can try to use these names to identify the FIPS where the weather event was reported.

Use the list of weather stations from Meteostat (https://github.com/meteostat/weather-stations?tab=readme-ov-file) to further identify locations

Note that lists of additional weather stations are available from NOAA: https://www.ncei.noaa.gov/access/homr/#. However, I've had trouble reliably identifying examples of weather stations from the downloaded lists, so sticking with meteostat for now.

In the code below, we'll load the Meteostat list, do some cleaning, add a geometry variable so we can merge it with the counties dataframe, and look up the FIPS code for each station from the counties data

In [10]:
#Load the Meteostat list of weather stations
import pandas as pd
df_meteostat_list = pd.read_json('../Data/full.json')

#Restrict the country variable to "US"
df_meteostat_list = df_meteostat_list[df_meteostat_list['country'] == "US"]

#Convert the name variable into strings and strip the text {'en': '
df_meteostat_list['name'] = df_meteostat_list['name'].astype(str).str.strip("{'en': '").str.strip("'").str.strip("'}")

#Extract the icao value from the identifiers variable
df_meteostat_list['icao'] = df_meteostat_list['identifiers'].apply(lambda x: x['icao'])

#Extract the latitude, longitude, and elevation from the location variable
df_meteostat_list['latitude'] = df_meteostat_list['location'].apply(lambda x: x['latitude'])
df_meteostat_list['longitude'] = df_meteostat_list['location'].apply(lambda x: x['longitude'])
df_meteostat_list['elevation'] = df_meteostat_list['location'].apply(lambda x: x['elevation'])

#Drop the identifiers, location, and inventory variables
df_meteostat_list = df_meteostat_list.drop(columns=['country','identifiers', 'location', 'inventory'])

#Convert the id, name, region, and icao variables into strings
df_meteostat_list['id'] = df_meteostat_list['id'].astype(str)
df_meteostat_list['name'] = df_meteostat_list['name'].astype(str)
df_meteostat_list['region'] = df_meteostat_list['region'].astype(str)
df_meteostat_list['icao'] = df_meteostat_list['icao'].astype(str)

#Convert df_meteostat_list into a geopandas dataframe
df_meteostat_list = geopandas.GeoDataFrame(df_meteostat_list, geometry=geopandas.points_from_xy(df_meteostat_list['longitude'], df_meteostat_list['latitude']))

#Set the CRS as EPSG 4269
# Note: I'm not entirely sure this is the CRS for the data - I couldn't find any info directly from the meteostat website
df_meteostat_list.set_crs(epsg=4269, inplace=True)

#Perform a spatial join between the counties and the df_meteostat_list dataframes to add the FIPS variable to df_meteostat_list
df_meteostat_list['FIPS'] = geopandas.sjoin(df_meteostat_list, counties, how='left', predicate='intersects')['FIPS']

Next we'll define a function that will look through the event narrative and look for mentions of weather stations, then find the corresponding FIPS code

Note that the next code chunk takes nearly 4 minutes to run.

In [11]:
def get_fips_from_meteostat(df_events_row):
    #Get the state from the row
    state = df_events_row['STATE_ABBREV']
    
    #Get the event narrative from the row
    narrative = df_events_row['EVENT_NARRATIVE']
    
    #Create a new dataframe from df_meteostat_list that has the same state
    df_meteostat_list_state = df_meteostat_list[df_meteostat_list['region'] == state]
    
    #Identify any word in the EVENT_NARRATIVE variable that matches the id variable, the icao variable, or the name variable in df_meteostat_list for the same STATE
    fips = []
    for index, row in df_meteostat_list_state.iterrows():
        if pd.notna(narrative):
            if str(row['name']) in narrative or str(row['icao']) in narrative or str(row['id']) in narrative:
                fips.append(row['FIPS'])
    return fips

#Apply the function to each row of df_events that lack latitude data and create a new variable that lists the fips values
df_events['FIPS_from_Meteostat'] = df_events[df_events['BEGIN_LAT'].isna()].apply(get_fips_from_meteostat, axis=1)

The NOAA data includes columns for STATE_FIPS and CZ_FIPS. The stat FIPS values appear correct, but the county FIPS values don't appear to match what I can find online.

For consistency, we can look up the FIPS for the county using the STATE_ABBREV and CZ_NAME variables

Note that this code takes 4 1/2 minutes to run

In [12]:
def get_fips_from_czname(row):
    #Get the state from the row
    state = row['STATE_ABBREV']
    
    #Get the county name from the row
    czname = row['CZ_NAME']
    
    #Create a new dataframe from counties that has the same state
    counties_state = counties[counties['STUSPS'] == state]
    
    fips = []
    for index, row in counties_state.iterrows():
        #if COUNTYNAME is not NaN and narrative is not NaN:
        if pd.notna(czname):
            #Convert row['NAME'] to all capitals
            uppername = row['NAME'].upper()
            if uppername in czname:
                fips.append(row['FIPS'])
    #print(fips)
    
    return fips

#Apply get_fips_from_narrative to each row of df_events and create a new variable that lists the fips values
# But do this only for rows of df_events that lack latitude and/or longitude data and only for rows of df_events that have a value for STATE_ABBREV
df_events['FIPS_from_CZNAME'] = df_events[df_events['BEGIN_LAT'].isna()].apply(get_fips_from_czname, axis=1)

Now, we'll create a list of all the FIPS associated with each event

In [13]:
#In the df_events data frame, create a new list of values by combining the values of the variables FIPS_from_Path, FIPS_from_Countyname_Narrative, FIPS_from_Meteostat, and FIPS_from_CZNAME, dropping duplicate values
df_events['FIPS'] = df_events[['FIPS_from_Path', 'FIPS_from_Countyname_Narrative', 'FIPS_from_Meteostat', 'FIPS_from_CZNAME']].apply(lambda x: list(set(x.dropna().sum())), axis=1)

There are several ways we could measure the severity of each event. One basic way is to compute the duration of the event based on the begin year/day/time (measured by the BEGIN_DATE_TIME variable) and end year/day/time (measured by the END_DATE_TIME variable). 

In general, we should be mindful of time zone differences. However, the NOAA data don't indicate which time zone the begin/end data are coming from. So the best we can do is assume that each event is confined to a single time zone, even for events that impact multiple counties.

In [21]:
#Create a new variable in df_events total_duration by subtracting the BEGIN_DATE_TIME from END_DATE_TIME, measuring in minutes.
#Specify the format day-month-year hour:minute:second
df_events['BEGIN_DATE_TIME'] = pd.to_datetime(df_events['BEGIN_DATE_TIME'], format='%d-%b-%y %H:%M:%S')
df_events['END_DATE_TIME'] = pd.to_datetime(df_events['END_DATE_TIME'], format='%d-%b-%y %H:%M:%S')
df_events['total_duration_min'] = (df_events['END_DATE_TIME'] - df_events['BEGIN_DATE_TIME']).dt.total_seconds() / 60.0

The NOAA data also includes several measures of the severity of the impact of the event: Injuries, Deaths, and Damage

In 2023, the US DoT's official value of a statistical life is $13.2 million. (see https://www.transportation.gov/office-policy/transportation-policy/revised-departmental-guidance-on-valuation-of-a-statistical-life-in-economic-analysis)

Information about statistical injuries are trickier to identify. We have some data from Switzerland:
https://ansperformance.eu/economics/cba/standard-inputs/chapters/value_of_a_statistical_injury.html
(It looks like the value of a life in Switzerland is only $4.3 million....)

Some data from 2013 - again, from Switzerland - that an injury was valued at $35k Swiss francs. Assuming 1.14 francs per USD. With inflation (data from https://www.inflationtool.com/euro-switzerland/2013-to-present-value), this would rise to 1.21 francs per USD in 2025. So the value of an injury in 2025 USD would be $42,350.

The various types of damage are already given in thousands of dollars.

We can combine these columns to get a compsite cost for each event.

In [23]:
#In the DAMAGE_PROPERTY and DAMAGE_CROPS variables, if the string ends with 'K', remove the 'K' at the end of the string, convert to an integer, and multiply by 1000. If the string ends with 'M', remove the 'M" at the end of the string, convert to an integer, and multiply by 1,000,000
def convert_damage(damage):
    if isinstance(damage, str):
        if damage.endswith('K'):
            return float(damage[:-1]) * 1000
        elif damage.endswith('M'):
            return float(damage[:-1]) * 1000000
    else:
        return 0

#Compute the total value of damage to property and crops
df_events['DAMAGE_PROPERTY_VALUE'] = df_events['DAMAGE_PROPERTY'].apply(convert_damage)
df_events['DAMAGE_CROPS_VALUE'] = df_events['DAMAGE_CROPS'].apply(convert_damage)

#Compute the total values of injuries and deaths
df_events['INJURIES_TOTAL_VALUE'] = (df_events['INJURIES_DIRECT'] + df_events['INJURIES_INDIRECT']) * 42350
df_events['DEATHS_TOTAL_VALUE'] = (df_events['DEATHS_DIRECT'] + df_events['DEATHS_INDIRECT']) * 13200000

#Compute the total value of damage, injuries, and deaths for each event
df_events['TOTAL_VALUE'] = df_events['DAMAGE_PROPERTY_VALUE'] + df_events['DAMAGE_CROPS_VALUE'] + df_events['INJURIES_TOTAL_VALUE'] + df_events['DEATHS_TOTAL_VALUE']

In [24]:
#Export df_events as a parquet file
df_events.to_parquet('../Data/df_events.parquet', index=False)